# First ADK Procurement Investigation

A coordinator delegates supplier-spend analysis to a single-turn Data Analyst, which uses one parameterized, read-only PostgreSQL tool. The sample tenant and records are fictional and local-only. A Google model call requires `GOOGLE_API_KEY`; the SQL tool can be validated without a model call.

## 1. Update Python Project for ADK Workflow

The application code lives in `src/ai_template_python`: ADK topology in `agents/`, the read-only query in `tools/sql/`, shared investigation contracts in `contracts.py`, and versioned state in `state.py`. The project dependency is locked in `pyproject.toml` and `uv.lock`.

In [11]:
from pathlib import Path
import tomllib

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = next(parent for parent in ROOT.parents if (parent / "pyproject.toml").is_file())

PROJECT = tomllib.loads((ROOT / "pyproject.toml").read_text(encoding="utf-8"))
assert "google-adk>=2.11.0,<3.0.0" in PROJECT["project"]["dependencies"]
for relative_path in (
    "src/ai_template_python/agents/first_investigation.py",
    "src/ai_template_python/tools/sql/spend_analysis.py",
    "src/ai_template_python/contracts.py",
    "src/ai_template_python/state.py",
):
    assert (ROOT / relative_path).is_file(), relative_path

print(f"Project ready: {PROJECT['project']['name']}")
print("ADK dependency and workflow modules are present.")

Project ready: ai-template-python
ADK dependency and workflow modules are present.


## 2. Install and Configure Google ADK in Notebook

Install the locked project and notebook tools with `uv sync --group notebook`, then select that environment as the Jupyter kernel. If using a non-project kernel, install the workspace package with `%pip install -e .` and restart the kernel. Set `GOOGLE_API_KEY` in the ignored root `.env`; never put the key in a notebook cell.

In [12]:
import os
from importlib.metadata import version

from dotenv import load_dotenv

load_dotenv(ROOT / ".env", override=False)

from google.adk import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

MODEL = os.getenv("GEMINI_MODEL", "gemini-flash-latest")
TENANT_ID = "00000000-0000-4000-8000-000000000001"  # trusted local synthetic tenant
APP_NAME = "enterprise_decision_intelligence"
USER_ID = "local-notebook-user"

print(f"Google ADK: {version('google-adk')}")
print(f"Model configured: {MODEL}")
print(f"GOOGLE_API_KEY present: {bool(os.getenv('GOOGLE_API_KEY'))}")
print(f"DATABASE_URL present: {bool(os.getenv('DATABASE_URL'))}")

Google ADK: 2.11.0
Model configured: gemini-flash-latest
GOOGLE_API_KEY present: True
DATABASE_URL present: True


## 3. Define Shared Models and Agent State

The Python package keeps contracts framework-independent and uses versioned JSON-serializable investigation state. This cell shows the same request and state shape that the SQL analyst handles.

In [13]:
from datetime import date

from ai_template_python.contracts import SpendAnalysisRequest
from ai_template_python.state import new_investigation_state

request = SpendAnalysisRequest(
    start_date=date(2024, 1, 1),
    end_date=date(2025, 12, 31),
    supplier_id="S102",
    group_by="month",
)
investigation_state = new_investigation_state(
    investigation_id="notebook-s102-spend",
    tenant_id=TENANT_ID,
    requester_id=USER_ID,
    question="Explain monthly supplier spend and the largest month-over-month change.",
)
print(request)
print({key: investigation_state[key] for key in ("schema_version", "status", "tenant_id", "question")})

SpendAnalysisRequest(start_date=datetime.date(2024, 1, 1), end_date=datetime.date(2025, 12, 31), supplier_id='S102', category_id=None, group_by='month')
{'schema_version': 1, 'status': 'queued', 'tenant_id': '00000000-0000-4000-8000-000000000001', 'question': 'Explain monthly supplier spend and the largest month-over-month change.'}


## 4. Implement Root Agent

The root coordinator delegates analytical work and returns a concise finding grounded in the specialist's output. Its instructions also prevent purchase-order spend from being misreported as paid invoice spend.

In [14]:
from ai_template_python.agents.first_investigation import root_agent

assert root_agent.name == "root_agent"
print(root_agent.name, root_agent.description)

root_agent Coordinates procurement investigations and presents evidence-based findings.


## 5. Implement Data Analyst Agent

The Data Analyst is a single-turn ADK sub-agent. It is responsible for selecting the safe analysis tool, interpreting returned figures, and clearly marking synthetic data.

In [15]:
from ai_template_python.agents.first_investigation import data_analyst_agent

assert data_analyst_agent.mode == "single_turn"
assert data_analyst_agent in root_agent.sub_agents
print(data_analyst_agent.name, data_analyst_agent.description)

data_analyst Analyzes tenant-scoped supplier purchase-order spend using SQL.


## 6. Create and Register SQL Tool

The analyst receives a Python function tool, not arbitrary SQL. SQL text is fixed in the module; supplier and date values are bound parameters, and tenant scope is injected from trusted ADK session state.

In [16]:
import inspect

from ai_template_python.tools.sql.spend_analysis import analyze_supplier_monthly_spend

tool_parameters = inspect.signature(analyze_supplier_monthly_spend).parameters
assert "supplier_id" in tool_parameters
assert "start_date" in tool_parameters and "end_date" in tool_parameters
assert "tenant_id" not in tool_parameters
assert "sql" not in tool_parameters
assert analyze_supplier_monthly_spend in data_analyst_agent.tools
print("Registered safe SQL tool:", analyze_supplier_monthly_spend.__name__)
print("LLM inputs:", [name for name in tool_parameters if name != "tool_context"])

Registered safe SQL tool: analyze_supplier_monthly_spend
LLM inputs: ['supplier_id', 'start_date', 'end_date']


## 7. Wire Root Agent to Data Analyst and SQL Tool

ADK registers the Data Analyst as a sub-agent and automatically provides delegation to the root. The Runner session receives the authorized local tenant ID as session state; it is not part of the user prompt or tool arguments.

In [17]:
session_service = InMemorySessionService()
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
)

print("Runner configured for:", root_agent.name)
print("Delegates:", [agent.name for agent in root_agent.sub_agents])

Runner configured for: root_agent
Delegates: ['data_analyst']


## 8. Run One End-to-End Investigation

Ask the analyst to summarize S102's 2024–2025 monthly GBP purchase-order spend and identify its largest absolute month-over-month change. The returned events include specialist/root handoffs and tool activity; live execution is skipped when either credential or database configuration is missing.

In [18]:
async def run_investigation(question: str) -> dict[str, object]:
    """Run one ADK investigation using a trusted local tenant session state."""
    if not os.getenv("GOOGLE_API_KEY"):
        raise RuntimeError("Set GOOGLE_API_KEY in the ignored root .env before a live model run.")
    if not os.getenv("DATABASE_URL"):
        raise RuntimeError("Set DATABASE_URL in the environment before a database investigation.")

    session = await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        state={"tenant_id": TENANT_ID},
    )
    message = types.Content(role="user", parts=[types.Part(text=question)])
    trace: dict[str, object] = {"agents": [], "tools": [], "errors": []}
    event_authors: list[str] = trace["agents"]  # type: ignore[assignment]
    tool_authors: list[str] = trace["tools"]  # type: ignore[assignment]

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session.id,
        new_message=message,
    ):
        if event.author not in event_authors:
            event_authors.append(event.author)
        if event.author not in {"root_agent", "data_analyst"} and event.author not in tool_authors:
            tool_authors.append(event.author)
        if event.content and event.content.parts:
            text = "".join(part.text or "" for part in event.content.parts)
            if text.strip():
                print(f"[{event.author}] {text.strip()}")

    return trace


async def run_with_free_tier_retry(question: str, attempts: int = 4) -> dict[str, object]:
    """Retry on Gemini free-tier 429/503 errors, waiting out the per-minute quota window."""
    import asyncio
    import re

    for attempt in range(1, attempts + 1):
        try:
            return await run_investigation(question)
        except Exception as error:  # noqa: BLE001
            message = str(error)
            if attempt == attempts or not any(code in message for code in ("429", "503")):
                raise
            match = re.search(r"retry in ([\d.]+)s", message)
            delay = float(match.group(1)) + 5 if match else 65
            print(f"Attempt {attempt} hit a free-tier limit; retrying in {delay:.0f}s")
            await asyncio.sleep(delay)

In [9]:
trace = await run_with_free_tier_retry(
    "summarize S102's 2024–2025 monthly GBP purchase-order spend and identify its largest absolute month-over-month change"
)
trace

/home/rajesh/projects/Portfolio/enterprise-decision-intelligence/.venv/lib/python3.13/site-packages/google/adk/models/llm_request.py:306: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


[data_analyst] ### Monthly Purchase-Order Spend Breakdown (GBP) – Supplier S102 (2024–2025)

*Note: The spend data returned for this period is **synthetic**.*

| Month | Spend (GBP) | MoM Change (GBP) | MoM Change (%) | PO Count |
| :--- | :--- | :--- | :--- | :--- |
| **2024-01** | £2,400.00 | — | — | 1 |
| **2024-02** | £2,204.40 | -£195.60 | -8.15% | 1 |
| **2024-03** | £1,204.80 | -£999.60 | -45.35% | 1 |
| **2024-04** | £2,313.80 | **+£1,109.00** | **+92.05%** | 1 |
| **2024-05** | £1,990.80 | -£323.00 | -13.96% | 1 |
| **2024-06** | £1,818.00 | -£172.80 | -8.68% | 1 |
| **2024-07** | £1,087.90 | -£730.10 | -40.16% | 1 |
| **2024-08** | £2,104.05 | +£1,016.15 | +93.40% | 1 |
| **2024-09** | £1,371.60 | -£732.45 | -34.81% | 1 |
| **2024-10** | £1,908.75 | +£537.15 | +39.16% | 1 |
| **2024-11** | £2,371.50 | +£462.75 | +24.24% | 1 |
| **2024-12** | £1,814.05 | -£557.45 | -23.51% | 1 |
| **2025-01** | £1,254.40 | -£559.65 | -30.85% | 1 |
| **2025-02** | £1,385.10 | +£130.70 | +10.42%

[root_agent] ### Summary of Findings: Supplier S102 Purchase-Order Spend (2024–2025)

Based on the evidence returned by the spend data analysis, below is the monthly purchase-order spend breakdown for supplier **S102** across 2024 and 2025.

---

### Monthly Spend & Month-over-Month (MoM) Changes

*All monetary figures are in **GBP (£)**.*

| Month | Spend (GBP) | MoM Change (GBP) | MoM Change (%) | PO Count |
| :--- | :---: | :---: | :---: | :---: |
| **2024-01** | £2,400.00 | — | — | 1 |
| **2024-02** | £2,204.40 | -£195.60 | -8.15% | 1 |
| **2024-03** | £1,204.80 | -£999.60 | -45.35% | 1 |
| **2024-04** | £2,313.80 | **+£1,109.00** | **+92.05%** | 1 |
| **2024-05** | £1,990.80 | -£323.00 | -13.96% | 1 |
| **2024-06** | £1,818.00 | -£172.80 | -8.68% | 1 |
| **2024-07** | £1,087.90 | -£730.10 | -40.16% | 1 |
| **2024-08** | £2,104.05 | +£1,016.15 | +93.40% | 1 |
| **2024-09** | £1,371.60 | -£732.45 | -34.81% | 1 |
| **2024-10** | £1,908.75 | +£537.15 | +39.16% | 1 |
| **2024-11** | £2

{'agents': ['root_agent', 'data_analyst'], 'tools': [], 'errors': []}